# 💰 03 — Funnel de conversion

Du `page_view` à l'achat, identifier les drop-offs et calculer le ROAS.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_sessions, load_events, load_transactions
from src.preprocessing import enrich
from src.datamarts import (
    build_conversion_funnel, build_campaign_performance, build_top_products
)
from src.utils import set_style

set_style()
sessions = enrich(load_sessions(), load_transactions())
events = load_events()
trans = load_transactions()

## Funnel

In [ ]:
funnel = build_conversion_funnel(sessions, events)
funnel

💡 **Insight clé** : la chute massive est entre `add_to_cart` et `begin_checkout`. C'est le pain point n°1 à investiguer (UX du panier ? Coûts cachés ? Frictions au login ?)

## ROAS par campagne

In [ ]:
campaigns = build_campaign_performance(sessions)
campaigns

In [ ]:
# Top 5 campagnes rentables (ROAS > 1)
profitable = campaigns[campaigns['roas'] > 1].sort_values('roas', ascending=False)
print('Campagnes rentables :')
profitable.head()

In [ ]:
# Campagnes en perte
losing = campaigns[campaigns['roas'] < 1].sort_values('cost_usd', ascending=False)
print('Campagnes en perte (ROAS < 1) — à investiguer ou couper :')
losing.head()

## Top produits

In [ ]:
build_top_products(trans).head(10)

## Calcul du CAC moyen

In [ ]:
paid_sessions = sessions[sessions['session_cost_usd'] > 0]
total_cost = paid_sessions['session_cost_usd'].sum()
total_paid_conversions = paid_sessions['converted'].sum()
cac = total_cost / max(total_paid_conversions, 1)
print(f'Total spent on paid : ${total_cost:,.2f}')
print(f'Conversions from paid : {total_paid_conversions}')
print(f'CAC (Customer Acquisition Cost) : ${cac:.2f}')